# Pipeline de Ingesta y Carga de Datos - Ecobici CDMX (2025 - 2026)

Este notebook descarga (si no existen localmente) y consolida los viajes de Ecobici exclusivamente para los años **2025** y **2026** utilizando Polars con lectura lazy optimizada, estandarización de columnas y unificación de tipos de datos xd.

Deje 2025 porque todavía no estan los datos completos de 2026, pero en caso que no se use se pueden borrar. 

In [1]:
# 1. IMPORTS Y CONFIGURACIÓN
from pathlib import Path
import re
import time
from typing import Dict, List, Optional, Tuple, Union

import polars as pl
import requests

print(f"Polars version: {pl.__version__}")

Polars version: 2.0.0


## 1.1 Descarga de datos de Ecobici (2025 y 2026)

In [2]:
# 2. DESCARGA AUTOMATIZADA (2025 y 2026)
URLS_POR_ANIO: Dict[int, List[str]] = {
    2025: [
        "https://ecobici.cdmx.gob.mx/wp-content/uploads/2025/02/2025-01.csv",
        "https://ecobici.cdmx.gob.mx/wp-content/uploads/2025/03/2025-02.csv",
        "https://ecobici.cdmx.gob.mx/wp-content/uploads/2025/04/2025-03.csv",
        "https://ecobici.cdmx.gob.mx/wp-content/uploads/2025/05/2025-04.csv",
        "https://ecobici.cdmx.gob.mx/wp-content/uploads/2025/06/2025-05.csv",
        "https://ecobici.cdmx.gob.mx/wp-content/uploads/2025/07/2025-06.csv",
        "https://ecobici.cdmx.gob.mx/wp-content/uploads/2025/08/2025-07.csv",
        "https://ecobici.cdmx.gob.mx/wp-content/uploads/2025/09/2025-08.csv",
        "https://ecobici.cdmx.gob.mx/wp-content/uploads/2025/10/2025-09.csv",
        "https://ecobici.cdmx.gob.mx/wp-content/uploads/2025/11/2025-10-1.csv",
        "https://ecobici.cdmx.gob.mx/wp-content/uploads/2025/12/2025-11.csv",
        "https://ecobici.cdmx.gob.mx/wp-content/uploads/2026/01/2025-12.csv",
    ],
    2026: [
        "https://ecobici.cdmx.gob.mx/wp-content/uploads/2026/02/2026-01.csv",
        "https://ecobici.cdmx.gob.mx/wp-content/uploads/2026/03/2026-02.csv",
        "https://ecobici.cdmx.gob.mx/wp-content/uploads/2026/04/2026-03.csv",
        "https://ecobici.cdmx.gob.mx/wp-content/uploads/2026/05/2026-04.csv",
        "https://ecobici.cdmx.gob.mx/wp-content/uploads/2026/06/2026-05.csv",
        "https://ecobici.cdmx.gob.mx/wp-content/uploads/2026/07/public_data_web_2026-06.csv",
        "https://ecobici.cdmx.gob.mx/wp-content/uploads/2026/08/public_data_web_2026-07.csv",
        "https://ecobici.cdmx.gob.mx/wp-content/uploads/2026/09/public_data_web_2026-08_2.csv",
        "https://ecobici.cdmx.gob.mx/wp-content/uploads/2026/10/public_data_web_2026-09.csv",
    ],
}

def descargar_datos_ecobici(
    anios: List[int] = [2025, 2026],
    destino: Union[str, Path] = "data",
    dormir: float = 0.5,
    forzar: bool = False
) -> List[Path]:
    """Descarga los archivos CSV de los años indicados si no existen en destino."""
    destino = Path(destino)
    destino.mkdir(parents=True, exist_ok=True)
    descargados: List[Path] = []

    for anio in sorted(anios):
        urls = URLS_POR_ANIO.get(anio, [])
        if not urls:
            print(f"[AVISO] No hay URLs configuradas para el año {anio}.")
            continue

        print(f"\nVerificando datos del año {anio} ({len(urls)} archivos)...\n")
        for url in urls:
            nombre_archivo = url.rstrip("/").split("/")[-1]
            ruta = destino / nombre_archivo

            if ruta.exists() and not forzar:
                size_mb = ruta.stat().st_size / (1024 * 1024)
                print(f"  [EXISTE]     {nombre_archivo:<32} ({size_mb:.1f} MB)")
                continue

            print(f"  [DESCARGA]   {nombre_archivo}... ", end="", flush=True)
            try:
                r = requests.get(url, stream=True, timeout=120)
                r.raise_for_status()
                with open(ruta, "wb") as f:
                    for chunk in r.iter_content(chunk_size=65536):
                        f.write(chunk)
                size_mb = ruta.stat().st_size / (1024 * 1024)
                print(f"OK ({size_mb:.1f} MB)")
                descargados.append(ruta)
                time.sleep(dormir)
            except Exception as e:
                print(f"ERROR: {e}")

    return descargados

## 1.2 Búsqueda y consolidación con Polars (2025 y 2026)

In [3]:
# 3. CARGA Y PROCESAMIENTO LAZY CON POLARS
COLUMN_MAP = {
    "ciclo_estacion_retiro": "ciclo_estacion_retiro",
    "ciclo_estacionretiro": "ciclo_estacion_retiro",
    "estacion_retiro": "ciclo_estacion_retiro",
    "ciclo_estacion_arribo": "ciclo_estacion_arribo",
    "ciclo_estacionarribo": "ciclo_estacion_arribo",
    "ciclo_estacion_arrivo": "ciclo_estacion_arribo",
    "estacion_arribo": "ciclo_estacion_arribo",
    "fecha_retiro": "fecha_retiro",
    "hora_retiro": "hora_retiro",
    "fecha_arribo": "fecha_arribo",
    "fecha_arrivo": "fecha_arribo",
    "hora_arribo": "hora_arribo",
    "hora_arrivo": "hora_arribo",
    "bici": "bici",
    "bicicleta": "bici",
    "genero_usuario": "genero_usuario",
    "genero": "genero_usuario",
    "edad_usuario": "edad_usuario",
    "edad": "edad_usuario",
}

def normalizar_columna(col: str) -> str:
    """Normaliza nombres de columnas a snake_case estandarizado."""
    c = str(col).strip().replace('"', "").replace(" ", "_").lower()
    return COLUMN_MAP.get(c, c)

def extraer_anio_mes(path: Path) -> Optional[Tuple[int, int]]:
    """Extrae año y mes a partir del nombre del archivo."""
    m = re.search(r"(20\d{2})[-_](\d{2})", path.name)
    if m:
        return int(m.group(1)), int(m.group(2))
    m_anio = re.search(r"(20\d{2})", path.name)
    if m_anio:
        return int(m_anio.group(1)), 99
    return None

def buscar_csvs(data_dir: Union[str, Path] = "data", anios_validos: Tuple[int, ...] = (2025, 2026)) -> List[Path]:
    """Filtra y ordena los CSVs locales pertenecientes exclusivamente a los años solicitados."""
    archivos = []
    for p in Path(data_dir).rglob("*.csv"):
        info = extraer_anio_mes(p)
        if info and info[0] in anios_validos:
            archivos.append((info[0], info[1], p.name, p))

    archivos.sort(key=lambda item: (item[0], item[1], item[2]))
    return [p for *_, p in archivos]

def cargar_y_unir_polars(
    data_dir: Union[str, Path] = "data",
    anios_validos: Tuple[int, ...] = (2025, 2026),
    verbose: bool = True
) -> pl.LazyFrame:
    """Construye un LazyFrame unificado aplicando normalización de columnas y casteo seguro de tipos."""
    archivos = buscar_csvs(data_dir=data_dir, anios_validos=anios_validos)
    if verbose:
        print(f"Archivos encontrados ({anios_validos}): {len(archivos)}")

    if not archivos:
        raise FileNotFoundError(f"No se encontraron archivos para los años {anios_validos} en '{data_dir}'.")

    lazy_frames: List[pl.LazyFrame] = []
    columnas_str = {"ciclo_estacion_retiro", "ciclo_estacion_arribo", "bici", "genero_usuario"}

    for f in archivos:
        try:
            if verbose:
                print(f"  -> Escaneando: {f.name:<32}", end="", flush=True)

            # Lectura inicial como texto para prevenir fallos por inconsistencias entre CSVs
            lf = pl.scan_csv(
                f,
                infer_schema_length=0,
                ignore_errors=True,
                null_values=["", "NULL", "NA", "N/A", "null", "\\N"]
            )

            # Normalización de encabezados
            cols = lf.collect_schema().names()
            renames = {c: normalizar_columna(c) for c in cols if c != normalizar_columna(c)}
            if renames:
                lf = lf.rename(renames)

            schema = lf.collect_schema()
            exprs_cast = []
            for col_name in columnas_str:
                if col_name in schema:
                    exprs_cast.append(pl.col(col_name).cast(pl.Utf8))

            if "edad_usuario" in schema:
                exprs_cast.append(
                    pl.col("edad_usuario").cast(pl.Float64, strict=False).cast(pl.Int32, strict=False)
                )

            if exprs_cast:
                lf = lf.with_columns(exprs_cast)

            # Incorporar metadatos de año y mes de origen
            info = extraer_anio_mes(f)
            anio_val, mes_val = info if info else (9999, 99)
            lf = lf.with_columns([
                pl.lit(anio_val).alias("anio_origen"),
                pl.lit(mes_val).alias("mes_origen"),
                pl.lit(f.name).alias("origen_archivo"),
            ])

            lazy_frames.append(lf)
            if verbose:
                print(" OK")
        except Exception as e:
            if verbose:
                print(f" ERROR: {e}")
            continue

    if not lazy_frames:
        raise ValueError("No se pudo cargar ningún archivo CSV válido.")

    lf_total = pl.concat(lazy_frames, how="diagonal_relaxed")
    if verbose:
        print("\nLazyFrame consolidado correctamente.")
    return lf_total

## 1.3 Ejecución del pipeline y obtención del DataFrame final (`df`)

In [4]:
# 1. Descargar archivos faltantes de 2025 y 2026
descargar_datos_ecobici(anios=[2025, 2026], destino="data")

# 2. Cargar LazyFrame unificado sólo para 2025 y 2026
lf = cargar_y_unir_polars(data_dir="data", anios_validos=(2025, 2026))

# 3. Materializar en memoria el DataFrame final df
df = lf.collect()

print(f"\nDataFrame materializado exitosamente: {df.shape[0]:,} filas x {df.shape[1]} columnas")
df.head()


Verificando datos del año 2025 (12 archivos)...

  [EXISTE]     2025-01.csv                      (105.8 MB)
  [EXISTE]     2025-02.csv                      (8.2 MB)
  [EXISTE]     2025-03.csv                      (107.2 MB)
  [EXISTE]     2025-04.csv                      (101.1 MB)
  [EXISTE]     2025-05.csv                      (104.0 MB)
  [EXISTE]     2025-06.csv                      (92.3 MB)
  [EXISTE]     2025-07.csv                      (97.3 MB)
  [EXISTE]     2025-08.csv                      (94.2 MB)
  [EXISTE]     2025-09.csv                      (92.8 MB)
  [EXISTE]     2025-10-1.csv                    (102.6 MB)
  [EXISTE]     2025-11.csv                      (93.6 MB)
  [EXISTE]     2025-12.csv                      (84.4 MB)

Verificando datos del año 2026 (9 archivos)...

  [EXISTE]     2026-01.csv                      (89.1 MB)
  [EXISTE]     2026-02.csv                      (89.4 MB)
  [EXISTE]     2026-03.csv                      (96.6 MB)
  [EXISTE]     2026-04.csv 

genero_usuario,edad_usuario,bici,ciclo_estacion_retiro,fecha_retiro,hora_retiro,ciclo_estacion_arribo,fecha_arribo,hora_arribo,anio_origen,mes_origen,origen_archivo
str,i32,str,str,str,str,str,str,str,i32,i32,str
"""M""",26,"""5180930""","""568""","""31/12/2024""","""23:57:02""","""572""","""01/01/2025""","""00:00:03""",2025,1,"""2025-01.csv"""
"""F""",54,"""3653953""","""283""","""31/12/2024""","""23:51:40""","""596""","""01/01/2025""","""00:00:41""",2025,1,"""2025-01.csv"""
"""M""",38,"""7511322""","""034""","""31/12/2024""","""23:48:36""","""064""","""01/01/2025""","""00:00:59""",2025,1,"""2025-01.csv"""
"""M""",41,"""3804572""","""258""","""31/12/2024""","""23:54:11""","""023""","""01/01/2025""","""00:01:08""",2025,1,"""2025-01.csv"""
"""M""",35,"""3848405""","""043""","""31/12/2024""","""23:35:28""","""126""","""01/01/2025""","""00:01:17""",2025,1,"""2025-01.csv"""
